# NB-3: Ablation Study — 5 Forgetting Strategies
**Publication blockers PB-11 + PB-12**

Compares 5 forgetting strategies side-by-side:
1. **No-Forgetting** — memory grows unbounded (baseline)
2. **LRU** — least recently used (classic baseline)
3. **Importance** — forget least important first
4. **CA-Formula-Only** — consolidation formula, gate disabled (θ=0) — **new ablation**
5. **Consolidation-Aware (Ours)** — full formula + gate (θ=0.3) — **our contribution**

Also reports **False Forgetting Rate (FFR)** per strategy — fraction of evicted
memories that were not yet consolidated (PB-12).

**Time estimate:** ~30–60 min (5 strategies × real LLM QA via Groq)

## Step 1 — Install & clone

In [ ]:
import os, subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers', 'hnswlib', 'python-dotenv', 'groq', 'requests'], check=True)

REPO = 'https://github.com/Lamaq-Mujpurwala/CSAM-IPD-HALH.git'
REPO_DIR = '/kaggle/working/CSAM-IPD-HALH' if os.path.exists('/kaggle') else '/content/CSAM-IPD-HALH'

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO, REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'csam_project'))
print(f'Ready in {os.getcwd()}')

## Step 2 — API key
**Kaggle:** Notebook → Settings → Secrets → `GROQ_API_KEY`

**Colab:** Left sidebar key icon → `GROQ_API_KEY`

In [ ]:
import os

def load_api_key():
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('GROQ_API_KEY')
        if key: os.environ['GROQ_API_KEY'] = key; return 'kaggle'
    except Exception: pass
    try:
        from google.colab import userdata
        key = userdata.get('GROQ_API_KEY')
        if key: os.environ['GROQ_API_KEY'] = key; return 'colab'
    except Exception: pass
    if os.environ.get('GROQ_API_KEY'): return 'env'
    raise RuntimeError('Add GROQ_API_KEY to Kaggle/Colab Secrets')

print(f'API key loaded from: {load_api_key()}')
with open('.env', 'w') as f:
    f.write(f"GROQ_API_KEY={os.environ['GROQ_API_KEY']}\n")

## Step 3 — Configure ablation run

In [ ]:
MODEL         = 'llama-3.1-8b-instant'
SEED          = 42
CONVERSATIONS = 5    # conversations to run (each generates QA pairs)
INTERACTIONS  = 50   # memory events per conversation (50=fast, 100=paper quality)
THRESHOLD     = 80   # forget trigger: evict when L2 > threshold memories

import os
safe_model = MODEL.replace('/', '_')
OUT_ABLATION = f'csam_project/evaluation/ablation_results_{safe_model}_s{SEED}.json'

print(f'Model:         {MODEL}')
print(f'Conversations: {CONVERSATIONS}')
print(f'Interactions:  {INTERACTIONS}')
print(f'Threshold:     {THRESHOLD}')
print(f'Output:        {OUT_ABLATION}')
print(f'\nEstimated API calls: ~{CONVERSATIONS * 10 * 5} (5 strategies)')

## Step 4 — Run ablation (5 strategies)
Runs all 5 strategies in sequence. Each uses the same conversations + QA pairs
so results are directly comparable. Progress is printed live.

In [ ]:
import subprocess, sys, os
os.chdir(REPO_DIR)

cmd = [
    sys.executable, '-m', 'csam_project.evaluation.run_ablation',
    '--conversations', str(CONVERSATIONS),
    '--interactions',  str(INTERACTIONS),
    '--threshold',     str(THRESHOLD),
    '--model',         MODEL,
    '--seed',          str(SEED),
    '--output',        OUT_ABLATION,
]
print('Starting ablation (5 strategies)...')
print('This may take 30-60 min depending on model speed.\n')
result = subprocess.run(cmd, capture_output=False, text=True)
print('\nAblation done' if result.returncode == 0 else 'Ablation FAILED — check output above')

## Step 5 — Display results table

In [ ]:
import json, os

if not os.path.exists(OUT_ABLATION):
    print(f'Output not found: {OUT_ABLATION}')
else:
    with open(OUT_ABLATION) as f:
        ab = json.load(f)

    print('=' * 80)
    print('ABLATION RESULTS — 5 Forgetting Strategies')
    print('=' * 80)
    print(f'{"Strategy":<35} {"F1":>7} {"Mem":>6} {"FFR":>7}')
    print('-' * 60)

    for r in ab.get('results', []):
        strategy = r['strategy']
        f1       = r.get('overall_f1', 0)
        mem      = r.get('memory_count', 0)
        ffr      = r.get('false_forgetting_rate', None)
        ffr_str  = f'{ffr:.3f}' if ffr is not None else 'N/A'
        marker   = ' ← OURS' if 'Consolidation-Aware (Ours)' in strategy else ''
        print(f'{strategy:<35} {f1:>7.4f} {mem:>6} {ffr_str:>7}{marker}')

    print('\nFFR = False Forgetting Rate (lower is better for CA-Ours)')
    print('Goal: CA-Ours FFR ≈ 0, all baselines FFR > 0')

    if ab.get('api_usage'):
        u = ab['api_usage']
        print(f'\nAPI: {u.get("total_requests","?")} requests, '
              f'{u.get("total_tokens",0):,} tokens')

## Step 6 — (Optional) Multi-seed variance run
Runs the ablation across 5 seeds to compute variance. Only run after the single-seed
ablation above confirms results look correct. Takes ~5× longer.

In [ ]:
# Uncomment to run multi-seed variance
# SEEDS = [42, 123, 456, 789, 1337]
# 
# for seed in SEEDS:
#     out = f'csam_project/evaluation/ablation_{safe_model}_s{seed}.json'
#     cmd = [
#         sys.executable, '-m', 'csam_project.evaluation.run_ablation',
#         '--conversations', str(CONVERSATIONS),
#         '--interactions',  str(INTERACTIONS),
#         '--threshold',     str(THRESHOLD),
#         '--model',         MODEL,
#         '--seed',          str(seed),
#         '--output',        out,
#     ]
#     print(f'Running seed={seed}...')
#     subprocess.run(cmd, capture_output=False, text=True)
print('Variance run commented out by default. Uncomment SEEDS block to enable.')

## Step 7 — Save results

In [ ]:
import shutil

if os.path.exists('/kaggle'):
    dest = os.path.join('/kaggle/working', os.path.basename(OUT_ABLATION))
    if os.path.exists(OUT_ABLATION):
        shutil.copy(OUT_ABLATION, dest)
        print(f'Saved to Kaggle output: {dest}')
else:
    try:
        from google.colab import files
        if os.path.exists(OUT_ABLATION):
            files.download(OUT_ABLATION)
            print(f'Downloaded: {OUT_ABLATION}')
    except ImportError:
        print(f'File at: {OUT_ABLATION}')